# ML69_K2 — Accident Hotspot Clustering (Kaggle)

แนวคิด (unsupervised, ไม่ใช้ป้ายกำกับ):
1. แปลง lat/lon เป็นเมตร แล้วใช้ **DBSCAN** หาจุดเสี่ยงรูปทรงบาง/วงแหวน/กลุ่มแน่น (คลื่นไซน์, แนวโค้ง, วงแหวน, กลุ่มกลม)
2. รวมชิ้นส่วนแนวโค้งที่ขาดเป็นช่วง ๆ เข้าด้วยกัน
3. จุดเสี่ยง **ที่ซ้อนตำแหน่งเดียวกันแต่ต่างช่วงเวลา** → แยกด้วย GMM บน (x, y, เวลาแบบวงกลม)
4. จุดเสี่ยงกระจายกว้าง 2 กลุ่ม → EM: Gaussian 2 ตัว + พื้นหลัง noise แบบ uniform (ตัดสินด้วย posterior สูงสุด)
5. ที่เหลือ = `-1` (noise)

ผลบน Public LB ของเวอร์ชันนี้ ≈ **0.885**

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from scipy.spatial import cKDTree
from scipy.stats import multivariate_normal as mvn

# ---------------- Config ----------------
EPS, MIN_SAMPLES = 120, 15      # DBSCAN (เมตร)
GROW = 0                        # รัศมีดึงจุด noise รอบกลุ่มเข้ากลุ่ม (0 = ไม่ดึง; ให้ผลดีสุดบน LB)
TIME_W = 2.0                    # น้ำหนักฟีเจอร์เวลา (cos/sin) ตอนแยกกลุ่มซ้อนตามเวลา
BLOB_THR = None                 # None = ใช้ argmax ของ posterior; หรือใส่ 0.55 ฯลฯ
NOISE_BOX = (22000 * 20500)     # พื้นที่ที่ใช้ประมาณความหนาแน่น noise (ตร.ม.)
SEED = 0

In [ ]:
def find(name):
    c = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'./**/{name}', recursive=True)
    if not c: raise FileNotFoundError(name)
    return c[0]

df = pd.read_csv(find('data.csv'))
sample = pd.read_csv(find('sample_submission.csv'))
print(df.shape, df.id.is_unique)

# พิกัด -> เมตร (ศูนย์กลาง = ค่าเฉลี่ย)
lat0, lon0 = df.lat.mean(), df.lon.mean()
df['x'] = (df.lon - lon0) * 111320 * np.cos(np.radians(lat0))
df['y'] = (df.lat - lat0) * 110574

# เวลา HH:MM -> ชั่วโมง และฟีเจอร์วงกลม
hm = df.time_of_day.str.split(':', expand=True).astype(int)
df['h'] = hm[0] + hm[1] / 60
ang = 2 * np.pi * df.h.values / 24
T = np.c_[np.cos(ang), np.sin(ang)]
X = df[['x', 'y']].values

plt.figure(figsize=(7, 7)); plt.scatter(df.x, df.y, s=1, alpha=.4); plt.gca().set_aspect('equal'); plt.title('Accidents (m)'); plt.show()

## 1) โครงสร้างบาง/วงแหวน/กลุ่มแน่นด้วย DBSCAN

In [ ]:
lab = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit_predict(X)
print('DBSCAN clusters:', sorted(set(lab) - {-1}))

# แนวโค้งด้านล่างถูกแตกเป็นหลายชิ้น (ความหนาแน่นต่ำ) -> รวมชิ้นที่อยู่ y เฉลี่ยต่ำกว่า -5 กม.
arc = [c for c in set(lab) if c >= 0 and X[lab == c][:, 1].mean() < -5000]
for c in arc: lab[lab == c] = arc[0]

# (ตัวเลือก) ดึงจุด noise ที่อยู่ใกล้กลุ่มเข้ากลุ่ม
if GROW > 0:
    tree = cKDTree(X[lab >= 0]); lab_core = lab[lab >= 0]
    ni = np.where(lab == -1)[0]; dist, ix = tree.query(X[ni])
    lab[ni[dist < GROW]] = lab_core[ix[dist < GROW]]

sizes = pd.Series(lab[lab >= 0]).value_counts()
print(sizes.to_dict())

## 2) จุดเสี่ยงที่ซ้อนกันตามเวลา
สองกลุ่มใหญ่ที่สุดจาก DBSCAN คือสองพื้นที่ที่มีจุดเสี่ยง 2 จุดซ้อนกัน (กลางคืน/กลางวัน) — ตรวจแล้วว่ากลุ่มอื่นมีการกระจายเวลาเหมือน noise จึงไม่แยก

In [ ]:
out = lab.copy()
nxt = lab.max() + 1
big = sizes.index[:2]
for c in big:
    idx = np.where(lab == c)[0]
    Z = np.c_[X[idx] / X[idx].std(0), T[idx] * TIME_W]
    gm = GaussianMixture(2, n_init=5, random_state=SEED).fit(Z)
    p = gm.predict(Z)
    out[idx[p == 1]] = nxt; nxt += 1
    hrs = [np.degrees(np.arctan2(gm.means_[j, 3], gm.means_[j, 2])) / 15 % 24 for j in range(2)]
    print(f'cluster {c}: split {np.bincount(p)} , mean hours {np.round(hrs,1)}')

## 3) กลุ่มกระจายกว้าง 2 กลุ่ม: Gaussian 2 ตัว + noise uniform (EM)

In [ ]:
rest = np.where(out == -1)[0]; R = X[rest]
mus = np.array([[5500, 5500], [-5500, -4500]], float)      # จุดเริ่มต้นจากการดูภาพ
S = [np.eye(2) * 1500**2, np.eye(2) * 1500**2]
pi = np.array([.15, .15, .7])

for it in range(200):
    pr = np.c_[[pi[j] * mvn(mus[j], S[j]).pdf(R) for j in range(2)]].T
    pr = np.c_[pr, pi[2] * np.full(len(R), 1 / NOISE_BOX)]
    r = pr / pr.sum(1, keepdims=True)
    pi = r.mean(0)
    for j in range(2):
        w = r[:, j]
        mus[j] = (w[:, None] * R).sum(0) / w.sum()
        dd = R - mus[j]
        S[j] = (w[:, None, None] * np.einsum('ni,nj->nij', dd, dd)).sum(0) / w.sum()

print('weights', pi.round(3), '\ncenters', mus.round(0), '\nsd', [np.sqrt(np.diag(s)).round(0) for s in S])

for j in range(2):
    m = (r[:, j] > BLOB_THR) if BLOB_THR else (r.argmax(1) == j)
    out[rest[m]] = nxt; nxt += 1

## 4) ตรวจผลและบันทึก submission

In [ ]:
res = pd.Series(out)
print('clusters:', len(set(out) - {-1}), '| noise(-1):', (out == -1).sum())
print(res.value_counts().sort_index().to_dict())

m = out >= 0
plt.figure(figsize=(9, 9))
plt.scatter(df.x[~m], df.y[~m], s=1, c='lightgrey')
plt.scatter(df.x[m], df.y[m], s=3, c=out[m] % 20, cmap='tab20')
plt.gca().set_aspect('equal'); plt.title('Final clusters'); plt.show()

sub = pd.DataFrame({'id': df.id, 'cluster': out.astype(int)})
sub = sample[['id']].merge(sub, on='id', how='left')
assert sub.cluster.notna().all() and sub.id.is_unique and len(sub) == len(sample)
sub['cluster'] = sub.cluster.astype(int)
path = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working') else 'submission.csv'
sub.to_csv(path, index=False)
print('saved', path, sub.shape)
sub.head()

## หมายเหตุ
- ARI ไม่สนชื่อกลุ่ม แต่ noise ต้องเป็น `-1`
- ค่าที่ปรับได้อยู่ในเซลล์ Config: `GROW`, `BLOB_THR`, `EPS/MIN_SAMPLES`
- การรวมชิ้นแนวโค้งและจุดเริ่มต้นของ EM กลุ่มกว้างตั้งจากการดูภาพของชุดข้อมูลนี้โดยเฉพาะ
- Public LB ใช้ ~30% ของข้อมูล ความต่างระดับ 0.001 ไม่ควรถือเป็นสัญญาณที่แท้จริง